# Note:
- The data from the [Chicago Data Portal](https://data.cityofchicago.org/browse?category=Public+Safety&sortBy=most_accessed&page=1&pageSize=20) and Crime Data set were enriched using multiple datasets from the portal. We initially stored them in the PostgreSQL database to generate the enriched dataset by joining multiple datasets using the_geom, but we discovered inconsistencies in the police beat, district, and sector fields. All missing fields were determined using multiple fields to generate the most accurate information, but there may be errors during the data wrangling process.

- We designate the primary Crime dataset as the authoritative source of truth. To ensure consistency and address missing values, we perform internal imputation using data from other sources to fill corresponding NaN entries in location-based fields.

In [1]:
# import libraries
from platform import python_version
import sys
import time
import pandas as pd
import pyarrow as pa
import pyarrow.ipc as ipc
import numpy as np
import importlib
import re

# python source path
sys.path.append('../Src/')

# python
import utils
import geo
import geo_dict

# seed
SEED = 1776

# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Pyarrow": pa.__version__
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

# capture time
start = time.time()

   Library Version
0   Python  3.13.9
1   Pandas   2.3.3
2    NumPy   2.3.4
3  Pyarrow  22.0.0


## Read Data
- The Chicago Crime Data contains Crime, Arrest, IUCR, Neighborhood, and Police Beat datasets from the Chicago Crime Portal.

In [2]:
# display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# reset options
# pd.reset_option('display.max_columns')

In [3]:
# load using pyarrow for performance (crime data is joined between crime & neighborhood & police using geom)
df_crime = pd.read_csv("../Data/chicago_crimes_export.csv", engine="pyarrow", dtype_backend="pyarrow")
# copy
df_orig = df_crime.copy

In [4]:
df_crime.head()

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,year,updated_on,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,community_code,community_name,ca_community_area,location
0,HY522530,2015-12-02 18:00:00,013XX W 110TH PL,0486,BATTERY,DOMESTIC BATTERY SIMPLE,N,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE,f,t,2234,22,34,75,2015,2018-02-10 15:50:01,08B,60643,207706232.893,Morgan Park,"MOUNT GREENWOOD,MORGAN PARK",91877340.6988,22,3,2234,75,MORGAN PARK,91877340.6988,"(41.693109422,-87.655623969)"
1,HY527714,2015-12-02 18:00:00,005XX W DEMING PL,0810,THEFT,OVER $500,I,THEFT,OVER $500,OTHER,f,f,1935,19,43,7,2015,2018-02-10 15:50:01,06,60614,94460631.7277,Lincoln Park,LINCOLN PARK,67401451.3717,19,3,1935,7,LINCOLN PARK,88316400.4728,"(41.928320067,-87.642957119)"
2,HY529524,2015-12-02 18:00:00,001XX S CAMPBELL AVE,1330,CRIMINAL TRESPASS,TO LAND,N,CRIMINAL TRESPASS,TO LAND,"SCHOOL, PUBLIC, BUILDING",f,f,1125,11,2,28,2015,2018-02-10 15:50:01,26,60612,106718949.397,United Center,UNITED CENTER,32520512.7053,11,2,1125,28,NEAR WEST SIDE,158492466.554,"(41.879585586,-87.688817204)"
3,HY525992,2015-12-02 18:00:00,027XX W SUPERIOR ST,0930,MOTOR VEHICLE THEFT,THEFT / RECOVERY - AUTOMOBILE,I,MOTOR VEHICLE THEFT,THEFT/RECOVERY: AUTOMOBILE,STREET,f,f,1221,12,26,24,2015,2018-02-10 15:50:01,07,60612,106718949.397,Humboldt Park,HUMBOLDT PARK,125010425.593,12,2,1221,24,WEST TOWN,127562904.597,"(41.894732704,-87.69550026)"
4,HY526797,2015-12-02 18:00:00,035XX W FIFTH AVE,0810,THEFT,OVER $500,I,THEFT,OVER $500,CHURCH/SYNAGOGUE/PLACE OF WORSHIP,t,f,1133,11,28,27,2015,2018-02-10 15:50:01,06,60624,99418122.6738,Garfield Park,GARFIELD PARK,89976069.5947,11,3,1133,27,EAST GARFIELD PARK,53883220.8462,"(41.875933208,-87.714806571)"


## Describe Data

In [5]:
df_crime.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
case_number,8470050,8469443,HJ590004,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,8470050,NaN,NaN,NaN,2011-08-03 09:16:56,2001-01-01 00:00:00,2005-06-13 19:30:00,2010-06-06 09:19:00,2017-05-19 15:50:45,2025-12-24 00:00:00,NaN
block,8470050,65510,001XX N STATE ST,17067,NaN,NaN,NaN,NaN,NaN,NaN,NaN
iucr,8470050,418,0820,679332,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_description,8455087,32,THEFT,1790788,NaN,NaN,NaN,NaN,NaN,NaN,NaN
secondary_description,8455087,369,SIMPLE,996364,NaN,NaN,NaN,NaN,NaN,NaN,NaN
index_code,8455087,2,N,5004675,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_type,8470050,34,THEFT,1798858,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,8470050,569,SIMPLE,996364,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_description,8454712,218,STREET,2213499,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# duplicate record
df_crime[df_crime.case_number == 'HJ590004'].sort_values('updated_on')

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,year,updated_on,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,community_code,community_name,ca_community_area,location
4676877,HJ590004,2003-08-27 08:35:00,039XX S WALLACE ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,WAREHOUSE,t,f,925,9,11,61,2003,2022-09-19 15:41:05,01A,60609,213490324.899,New City,BACK OF THE YARDS,134636963.254,9,2,925,61,NEW CITY,134636963.254,"(41.822967958,-87.641048569)"
4676878,HJ590004,2003-08-27 08:35:00,039XX S WALLACE ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,WAREHOUSE,t,f,925,9,11,61,2003,2022-09-19 15:41:05,01A,60609,213490324.899,New City,BACK OF THE YARDS,134636963.254,9,2,925,61,NEW CITY,134636963.254,"(41.822967958,-87.641048569)"
4676879,HJ590004,2003-08-27 08:35:00,039XX S WALLACE ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,WAREHOUSE,t,f,925,9,11,61,2003,2022-09-19 15:41:05,01A,60609,213490324.899,New City,BACK OF THE YARDS,134636963.254,9,2,925,61,NEW CITY,134636963.254,"(41.822967958,-87.641048569)"
4676880,HJ590004,2003-08-27 08:35:00,039XX S WALLACE ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,WAREHOUSE,t,f,925,9,11,61,2003,2022-09-19 15:41:05,01A,60609,213490324.899,New City,BACK OF THE YARDS,134636963.254,9,2,925,61,NEW CITY,134636963.254,"(41.822967958,-87.641048569)"
4676881,HJ590004,2003-08-27 08:35:00,039XX S WALLACE ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,WAREHOUSE,t,f,925,9,11,61,2003,2022-09-19 15:41:05,01A,60609,213490324.899,New City,BACK OF THE YARDS,134636963.254,9,2,925,61,NEW CITY,134636963.254,"(41.822967958,-87.641048569)"
4676882,HJ590004,2003-08-27 08:35:00,039XX S WALLACE ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,WAREHOUSE,t,f,925,9,11,61,2003,2022-09-19 15:41:05,01A,60609,213490324.899,New City,BACK OF THE YARDS,134636963.254,9,2,925,61,NEW CITY,134636963.254,"(41.822967958,-87.641048569)"


## Data Wrangle
- Chicago's `IUCR` codes (Illinois Uniform Crime Reporting) are four-digit codes for classifying crimes, with the Chicago Police Department (CPD) using over 400, including FBI Index Offenses (homicide, robbery, theft) and Non-Index offenses (vandalism, weapons violations)
- Chicago has `50 wards`, each represented by an alderperson, with boundaries redrawn every eight years
- The Chicago Police Department (CPD) divides the city into `22 Districts`, which are further broken down into smaller patrol zones called `Beats`, with specific 4-digit numbers for each area
- Chicago is divided into `77 official Community Areas`

In [7]:
# Use a single Arrow string & int64 type instance to save memory
arrow_string = pd.ArrowDtype(pa.string())
arrow_int64 = pd.ArrowDtype(pa.int64())
arrow_bool = pd.ArrowDtype(pa.bool8())

In [8]:
# Convert & Force pyarrow
df_crime['community_name'] = df_crime['community_name'].str.title()
df_crime['community_name'] = df_crime['community_name'].astype(arrow_string)

In [9]:
# get dupes
dupes = df_crime.duplicated(keep='last')
# any duplicates
if dupes.any():
    print(f"Number of Duplicates: {dupes.sum():,}")
else:
    print("No Duplicates")

Number of Duplicates: 178


In [10]:
# num of rows before 
before = df_crime.shape[0]
# remove any duplicates
df_crime = df_crime.sort_values('updated_on').drop_duplicates(keep='last').reset_index(drop=True)
# num of rows after 
after = df_crime.shape[0]
print(f"Duplicate Rows Removed: {(before - after):,}")
print(f"Shape: {df_crime.shape[0]:,} Rows & {df_crime.shape[1]:,} Columns")

Duplicate Rows Removed: 178
Shape: 8,469,872 Rows & 31 Columns


In [11]:
# drop updated_on
df_crime = df_crime.drop(columns=['updated_on', 'case_number'])

In [12]:
# display number of unique values
for i in df_crime.columns:
    print(f"{i}: {df_crime[i].nunique():,}")

date: 3,545,853
block: 65,510
iucr: 418
primary_description: 32
secondary_description: 369
index_code: 2
primary_type: 34
description: 569
location_description: 218
arrest: 2
domestic: 2
beat: 305
district: 24
ward: 50
community_area: 78
year: 25
fbi_code: 26
zip_code: 59
zip_code_area: 59
primary_neighborhood: 98
secondary_neighborhood: 78
neighborhood_area: 98
p_district: 22
p_sector: 4
p_beat: 274
community_code: 77
community_name: 77
ca_community_area: 77
location: 910,551


#### New Feature(s)

In [13]:
# Add Month & Day of the Week
months = ['January', 'February', 'March', 'April', 'May', 'June', 
          'July', 'August', 'September', 'October', 'November', 'December']
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Extract integers and map them
# .dt.month returns 1-12, so we subtract 1 for 0-based indexing
df_crime['month'] = np.array(months)[df_crime['date'].dt.month.values - 1]

# .dt.dayofweek returns 0-6 (0 is Monday)
df_crime['day_of_week'] = np.array(days)[df_crime['date'].dt.dayofweek.values]

# convert to pyarrow
df_crime['month'] = df_crime['month'].astype(arrow_string)
df_crime['day_of_week'] = df_crime['day_of_week'].astype(arrow_string)

In [14]:
# Map to strings first
df_crime['quarter'] = df_crime['date'].dt.quarter.map({1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}).astype(arrow_string)
# combine
df_crime['year_quarter'] = (df_crime['year'].astype("string[pyarrow]") + "-" + df_crime['quarter'])
# Force pyarrow datatype
df_crime['year_quarter'] = df_crime['year_quarter'].astype(arrow_string)

| Interval (Inclusive, Exclusive) | Mathematical Notation | Label        | Hours Included    |
|--------------------------------|----------------------|--------------|-------------------|
| 1st: 0 to 4                    | \([0, 4)\)          | Late Night   | 0, 1, 2, 3       |
| 2nd: 4 to 8                    | \([4, 8)\)          | Early Morning| 4, 5, 6, 7       |
| 3rd: 8 to 12                   | \([8, 12)\)         | Morning      | 8, 9, 10, 11     |
| 4th: 12 to 16                  | \([12, 16)\)        | Afternoon    | 12, 13, 14, 15   |
| 5th: 16 to 20                  | \([16, 20)\)        | Evening      | 16, 17, 18, 19   |
| 6th: 20 to 24                  | \([20, 24)\)        | Night        | 20, 21, 22, 23   |

In [15]:
# Get the hours as a PyArrow-backed integer
hours = df_crime['date'].dt.hour.values

# Use np.digitize for ultra-fast binning (vectorized)
# bins: [0, 4, 8, 12, 16, 20, 24]
# digitize returns 1 for 0-3, 2 for 4-7, etc.
bin_indices = np.digitize(hours, bins=[4, 8, 12, 16, 20])

# Map indices to labels
time_labels = np.array(['Late Night', 'Early Morning', 'Morning', 'Afternoon', 'Evening', 'Night'])
df_crime['time_of_day'] = time_labels[bin_indices]

# Final cast to string[pyarrow]
df_crime['time_of_day'] = df_crime['time_of_day'].astype(arrow_string)

#### FBI Code Mapping

In [16]:
# display fbi_code
print(sorted(df_crime['fbi_code'].unique()))

['01A', '01B', '02', '03', '04A', '04B', '05', '06', '07', '08A', '08B', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '22', '24', '26']


In [17]:

# determine specific values in the data that are missing from the mapping dictionary
df_crime.loc[~df_crime["fbi_code"].isin(geo_dict.fbi_codes.keys()), "fbi_code" ].unique()

<ArrowExtensionArray>
[]
Length: 0, dtype: string[pyarrow]

In [18]:
df_crime[["fbi_code_desc", "fbi_index_code"]] = pd.DataFrame({
    "fbi_code_desc": df_crime["fbi_code"].map(lambda x: geo_dict.fbi_codes[x]["desc"]),
    "fbi_index_code": df_crime["fbi_code"].map(lambda x: geo_dict.fbi_codes[x]["is_index"])
})

# convert to arrow datatype
df_crime["fbi_code_desc"] = df_crime["fbi_code_desc"].astype(arrow_string)
df_crime["fbi_index_code"] = df_crime["fbi_index_code"].astype(arrow_bool)

# drop fbi_code & relared descriptions
df_crime = df_crime.drop(columns=['block', 'iucr', 'primary_description', 'secondary_description', 'primary_type', 'fbi_code', 'index_code'])

In [19]:
# df_crime[['iucr','primary_description','secondary_description','description','fbi_code']][df_crime['fbi_code'] == '01A'].sample(5)
df_crime[['fbi_code_desc','description', 'location_description', 'domestic', 'fbi_index_code']].sample(n=5, random_state=SEED)

,fbi_code_desc,description,location_description,domestic,fbi_index_code
2248216,Simple Assault,SIMPLE,STREET,f,0
2467513,Larceny – Theft,OVER $500,PARKING LOT/GARAGE(NON.RESID.),f,1
6123113,Drug Abuse Violations,POSS: CANNABIS 30GMS OR LESS,RESIDENCE,f,0
6472970,Miscellaneous Non-Index Offenses,VIOLATE ORDER OF PROTECTION,APARTMENT,t,0
5868892,Burglary,FORCIBLE ENTRY,COMMERCIAL / BUSINESS OFFICE,f,1


In [20]:
#### Neighborhood Compare

In [21]:
# 1. Create a boolean mask using the underlying Arrow arrays (fastest)
mask = df_crime['primary_neighborhood'].str.lower() != df_crime['secondary_neighborhood'].str.lower()

# 2. Slice, Drop Duplicates, and then Sort
# Reducing the rows BEFORE sorting is the key to speed.
diff_neighborhoods = (
    df_crime.loc[mask, ['ward','community_area','zip_code','beat','district','primary_neighborhood','secondary_neighborhood']]
    .drop_duplicates()
    .sort_values('primary_neighborhood')
)

# display
print(diff_neighborhoods.sample(n=10, random_state=SEED).sort_values('primary_neighborhood').to_string())

         ward  community_area  zip_code  beat  district primary_neighborhood       secondary_neighborhood
1613651    11              35     60616   211         2        Armour Square      ARMOUR SQUARE,CHINATOWN
493630      8              45     60617   414         4          Avalon Park  AVALON PARK,CALUMET HEIGHTS
4433315    25              33     60616  2111         9            Chinatown      ARMOUR SQUARE,CHINATOWN
50287      23              57     60632   815         8       Garfield Ridge               MIDWAY AIRPORT
4093381     3              40     60615   232         9      Grand Boulevard                  BRONZEVILLE
43508    <NA>            <NA>     60621   731         7       Grand Crossing  SOUTH SHORE, GRAND CROSSING
523032     31              20     60641  2524        25              Hermosa      BELMONT CRAIGIN,HERMOSA
531514     31              16     60641  1731        17              Hermosa      BELMONT CRAIGIN,HERMOSA
2739079     5              43     60619   322 

In [22]:
# 1. Create a boolean mask using the underlying Arrow arrays (fastest)
mask = df_crime['primary_neighborhood'].str.lower() != df_crime['community_name'].str.lower()

# 2. Slice, Drop Duplicates, and then Sort
# Reducing the rows BEFORE sorting is the key to speed.
diff_neighborhoods = (
    df_crime.loc[mask, ['ward','community_area', 'community_code','zip_code','beat','district','primary_neighborhood','community_name']]
    .drop_duplicates()
    .sort_values('primary_neighborhood')
)

# display
print(diff_neighborhoods.sample(n=10, random_state=SEED).sort_values('primary_neighborhood').to_string())

         ward  community_area  community_code  zip_code  beat  district primary_neighborhood      community_name
6246423    20              61              67     60609   934         9            Englewood      West Englewood
1000       28              26              26     60624  1122        11        Garfield Park  West Garfield Park
6245956    27              27              27     60612  1124        11        Garfield Park  East Garfield Park
4881        1              24              24     60647  1421        14        Humboldt Park           West Town
2398841    26              24              24     60622  1423      <NA>        Humboldt Park           West Town
43529    <NA>            <NA>              28     60607  1213        12    Little Italy, UIC      Near West Side
3942513    42               8               8     60654  1834         1          River North     Near North Side
7993016    44               6               8     60611  1924        18        Streeterville    

In [23]:
# drop secondary_neighborhood
df_crime = df_crime.drop(columns=['secondary_neighborhood'])
# display
df_crime.sample(n=5, random_state=SEED)

,date,description,location_description,arrest,domestic,beat,district,ward,community_area,year,zip_code,zip_code_area,primary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,community_code,community_name,ca_community_area,location,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code
2248216,2010-04-07 05:20:00,SIMPLE,STREET,t,f,1231,12,2,28,2010,60608,176505462.842,"Little Italy, UIC",71376244.1225,12,3,1233,28,Near West Side,158492466.554,"(41.865999363,-87.665853961)",April,Wednesday,Q2,2010-Q2,Early Morning,Simple Assault,0
2467513,2007-11-17 16:30:00,OVER $500,PARKING LOT/GARAGE(NON.RESID.),f,f,815,8,23,62,2007,60632,211755252.964,West Elsdon,32711091.4259,8,1,815,62,West Elsdon,32711091.4259,"(41.799808989,-87.723350952)",November,Saturday,Q4,2007-Q4,Evening,Larceny – Theft,1
6123113,2018-05-04 06:34:00,POSS: CANNABIS 30GMS OR LESS,RESIDENCE,t,f,1132,11,24,29,2018,60624,99418122.6738,North Lawndale,89487422.0244,11,3,1132,29,North Lawndale,89487422.0242,"(41.867071507,-87.726023529)",May,Friday,Q2,2018-Q2,Early Morning,Drug Abuse Violations,0
6472970,2019-08-25 05:02:00,VIOLATE ORDER OF PROTECTION,APARTMENT,t,t,631,6,8,44,2019,60619,167872012.337,Chatham,82320670.3112,6,3,631,44,Chatham,82320670.3112,"(41.746980538,-87.600214942)",August,Sunday,Q3,2019-Q3,Early Morning,Miscellaneous Non-Index Offenses,0
5868892,2002-07-06 22:00:00,FORCIBLE ENTRY,COMMERCIAL / BUSINESS OFFICE,f,f,2011,20,40,2,2002,60659,69698411.7811,West Ridge,98429094.8621,20,1,2011,2,West Ridge,98429094.8621,"(41.990474245,-87.69375814)",July,Saturday,Q3,2002-Q3,Night,Burglary,1


#### Datatype Change (Boolean)

In [24]:
# check for unique values
df_crime[['arrest','domestic']].apply(lambda s: s.unique())

,arrest,domestic
0,f,f
1,t,t


In [25]:
# convert to pyarrow boolean
df_crime[['arrest','domestic']] = (
    df_crime[['arrest','domestic']] # domestic: Domestic violence
        .apply(lambda col: col.map({'t': True, 'f': False}))
        .astype(arrow_bool)
)

In [26]:
# display null columns
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,872) ---
                       Count Percentage
ward                  614815    7.2588%
community_area        613682    7.2455%
p_district            116810    1.3791%
p_sector              116810    1.3791%
p_beat                116810    1.3791%
primary_neighborhood  116237    1.3724%
neighborhood_area     116237    1.3724%
community_code        116237    1.3724%
community_name        116237    1.3724%
ca_community_area     116237    1.3724%
zip_code              116181    1.3717%
zip_code_area         116181    1.3717%
location               94303    1.1134%
location_description   15338    0.1811%
district                  47    0.0006%


#### Feature Information:
* A Chicago `ward` is one of 50 legislative districts, each represented by an elected Alderman on the City Council, serving as local government branches to provide city services, manage development, and reflect community demographics, with boundaries redrawn every 10 years based on census data.
* The `district` feature refers to the city's 22 police districts, which are geographic areas used to organize crime data.
* The `beat` feature in Chicago crime data identifies the smallest geographic police area (a beat) where a crime occurred.
* The `sector` refers to a specific geographic division used by the Chicago Police Department (CPD), where several smaller `beats` (police patrol areas) are grouped together to form a sector, which then rolls up into a larger `district`, providing a layered geographic context for analyzing crime trends.
* The `Community Area` feature refers to one of 77 distinct, officially defined, and geographically stable neighborhoods used for urban planning and statistical analysis. This feature allows categorizing crime incidents by location, enabling trend analysis and identifying high-crime areas.

In [27]:
# Nans
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,872) ---
                       Count Percentage
ward                  614815    7.2588%
community_area        613682    7.2455%
p_district            116810    1.3791%
p_sector              116810    1.3791%
p_beat                116810    1.3791%
primary_neighborhood  116237    1.3724%
neighborhood_area     116237    1.3724%
community_code        116237    1.3724%
community_name        116237    1.3724%
ca_community_area     116237    1.3724%
zip_code              116181    1.3717%
zip_code_area         116181    1.3717%
location               94303    1.1134%
location_description   15338    0.1811%
district                  47    0.0006%


In [28]:
# display ward
utils.wrap_unique(df_crime, 'ward')

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42,
43, 44, 45, 46, 47, 48, 49, 50]
::::: Unique Count: 50 (+ 614,815 nulls)


In [29]:
# display district
utils.wrap_unique(df_crime, 'district')
utils.wrap_unique(df_crime, 'p_district')

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24,
25, 31]
::::: Unique Count: 24 (+ 47 nulls)
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 22, 24, 25]
::::: Unique Count: 22 (+ 116,810 nulls)


In [30]:
# https://www.chicagopolice.org/statistics-data/crime-statistics/
# sector is represented as Area
print(sorted(df_crime.p_sector.fillna(-1).unique()))

[-1, 1, 2, 3, 5]


In [31]:
# display primary_neighborhood
utils.wrap_unique(df_crime, 'primary_neighborhood')

[Albany Park, Andersonville, Archer Heights, Armour Square, Ashburn, Auburn
Gresham, Austin, Avalon Park, Avondale, Belmont Cragin, Beverly, Boystown,
Bridgeport, Brighton Park, Bucktown, Burnside, Calumet Heights, Chatham, Chicago
Lawn, Chinatown, Clearing, Douglas, Dunning, East Side, East Village, Edgewater,
Edison Park, Englewood, Fuller Park, Gage Park, Galewood, Garfield Park,
Garfield Ridge, Gold Coast, Grand Boulevard, Grand Crossing, Grant Park,
Greektown, Hegewisch, Hermosa, Humboldt Park, Hyde Park, Irving Park, Jackson
Park, Jefferson Park, Kenwood, Lake View, Lincoln Park, Lincoln Square, Little
Italy, UIC, Little Village, Logan Square, Loop, Lower West Side, Magnificent
Mile, Mckinley Park, Millenium Park, Montclare, Morgan Park, Mount Greenwood,
Museum Campus, Near South Side, New City, North Center, North Lawndale, North
Park, Norwood Park, O'Hare, Oakland, Old Town, Portage Park, Printers Row,
Pullman, River North, Riverdale, Rogers Park, Roseland, Rush & Division,
Sau

In [32]:
# display community_name
utils.wrap_unique(df_crime, 'community_name')

[Albany Park, Archer Heights, Armour Square, Ashburn, Auburn Gresham, Austin,
Avalon Park, Avondale, Belmont Cragin, Beverly, Bridgeport, Brighton Park,
Burnside, Calumet Heights, Chatham, Chicago Lawn, Clearing, Douglas, Dunning,
East Garfield Park, East Side, Edgewater, Edison Park, Englewood, Forest Glen,
Fuller Park, Gage Park, Garfield Ridge, Grand Boulevard, Greater Grand Crossing,
Hegewisch, Hermosa, Humboldt Park, Hyde Park, Irving Park, Jefferson Park,
Kenwood, Lake View, Lincoln Park, Lincoln Square, Logan Square, Loop, Lower West
Side, Mckinley Park, Montclare, Morgan Park, Mount Greenwood, Near North Side,
Near South Side, Near West Side, New City, North Center, North Lawndale, North
Park, Norwood Park, Oakland, Ohare, Portage Park, Pullman, Riverdale, Rogers
Park, Roseland, South Chicago, South Deering, South Lawndale, South Shore,
Uptown, Washington Heights, Washington Park, West Elsdon, West Englewood, West
Garfield Park, West Lawn, West Pullman, West Ridge, West Town, W

In [33]:
# Determine if one is missing, the other is not
mask = df_crime.primary_neighborhood.isna() ^ df_crime.community_name.isna()
df_crime.loc[mask, ['primary_neighborhood', 'community_name']].head()

,primary_neighborhood,community_name


In [34]:
# display community_area
utils.wrap_unique(df_crime, 'community_area')

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21,
22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41,
42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61,
62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77]
::::: Unique Count: 78 (+ 613,682 nulls)


In [35]:
# display community_code
utils.wrap_unique(df_crime, 'community_code')

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42,
43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62,
63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77]
::::: Unique Count: 77 (+ 116,237 nulls)


In [36]:
mask = df_crime.community_area.eq(0)
df_crime.loc[mask,].shape[0]

76

In [37]:
# Remove NaNs & remove duplicates & sort by zip_code
df_crime.loc[mask, ['community_area', 'community_code', 'zip_code','primary_neighborhood', 'community_name']]\
    .dropna().sort_values('zip_code').drop_duplicates()

,community_area,community_code,zip_code,primary_neighborhood,community_name
2504104,0,32,60602,Loop,Loop
2349620,0,32,60603,Loop,Loop
725263,0,56,60638,Garfield Ridge,Garfield Ridge
4521377,0,75,60655,Morgan Park,Morgan Park
1815286,0,76,60656,O'Hare,Ohare
2509159,0,10,60656,Norwood Park,Norwood Park
3617440,0,76,60666,O'Hare,Ohare


In [38]:
# build lookup: community_name → community_code (only valid rows)
lookup = (
    df_crime.loc[df_crime.community_area.ne(0), ['community_name', 'community_code']]
        .dropna().drop_duplicates()
        .set_index('community_name')['community_code'] # Set the index on community_name, display community_area column.
)

# fill zero community_area using the lookup & NaNs not in the lookup table
df_crime.loc[mask, 'community_area'] = (df_crime.loc[mask, 'community_name'].map(lookup))

In [39]:
# check
# all the '0' are updated
print(f"::::::: Community_area count: {df_crime.community_area.eq('0').sum()}\n")

df_crime.loc[mask, ['community_area', 'community_code', 'zip_code','primary_neighborhood', 'community_name']] \
    .dropna().sort_values('zip_code').drop_duplicates()



::::::: Community_area count: 0



,community_area,community_code,zip_code,primary_neighborhood,community_name
2504104,32,32,60602,Loop,Loop
2349620,32,32,60603,Loop,Loop
725263,56,56,60638,Garfield Ridge,Garfield Ridge
4521377,75,75,60655,Morgan Park,Morgan Park
1815286,76,76,60656,O'Hare,Ohare
2509159,10,10,60656,Norwood Park,Norwood Park
3617440,76,76,60666,O'Hare,Ohare


**Note:**
* Chicago’s crime data is recorded across a complex framework of overlapping jurisdictions, ranging from political districts to social neighborhoods. At the administrative level, the Chicago Police Department operates through a hierarchy of Districts and Beats. A Beat is the smallest geographic unit, assigned to a specific patrol car for community policing, while multiple Beats are grouped into a District managed by a central precinct. For example, Beats 2511, 2514, and 2521 all fall under the jurisdiction of District 025. Because these boundaries are drawn based on population density and response times rather than cultural history, they rarely align perfectly with the city’s social fabric.
* To provide a more stable lens for analysis, researchers utilize the city’s 77 Community Areas. Established in the 1920s by the University of Chicago, these fixed boundaries remain unchanged by political redistricting or postal updates, allowing for consistent longitudinal tracking of crime trends over decades. In contrast, Chicago’s 50 Wards are political entities redrawn every ten years to ensure equal population representation. Because Wards are subject to frequent shifts, they often bifurcate cohesive community areas and primary neighborhoods.
* Ultimately, "neighborhood" designations like "Albany Park" or "Irving Park" reflect social and historical identities rather than law-enforcement jurisdictions. Because these residential areas are often too large for a single patrol car to cover, a single neighborhood is frequently split across multiple Police Beats. This misalignment means that a single criminal incident may be categorized differently depending on whether the analyst is looking through a political (Ward), statistical (Community Area), or operational (Police District) lens.

**NaN Note:**
- We designate the primary Crime dataset as the authoritative source of truth. To ensure consistency and address missing values, we perform internal imputation by using the data from other sources to fill corresponding NaN entries within the location-based fields.

In [40]:
df_crime.head()

,date,description,location_description,arrest,domestic,beat,district,ward,community_area,year,zip_code,zip_code_area,primary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,community_code,community_name,ca_community_area,location,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code
0,2001-12-30 02:30:00,AUTOMOBILE,STREET,0,0,334,3,7,43,2001,60649,80526075.8505,South Shore,81812716.3904,3,3,334,43,South Shore,81812716.3958,"(41.76145747,-87.566281678)",December,Sunday,Q4,2001-Q4,Late Night,Motor Vehicle Theft,1
1,2001-12-24 12:30:00,FRAUD OR CONFIDENCE GAME,APARTMENT,0,0,2313,19,46,3,2001,60640,77305245.6358,Uptown,65095642.836,19,1,1914,3,Uptown,65095642.7289,"(41.965220666,-87.650159176)",December,Monday,Q4,2001-Q4,Afternoon,Fraud,0
2,2001-12-31 23:00:00,FINANCIAL ID THEFT:$300 &UNDER,APARTMENT,0,0,1531,15,37,25,2001,60651,99039621.5713,Austin,170037750.826,15,3,1531,25,Austin,199254203.427,"(41.897955908,-87.754304189)",December,Monday,Q4,2001-Q4,Night,Larceny – Theft,1
3,2001-12-23 22:30:00,AUTOMOBILE,STREET,0,0,1022,10,12,30,2001,60623,155285530.844,Little Village,127998297.819,10,2,1022,30,South Lawndale,127998297.867,"(41.855402607,-87.698559781)",December,Sunday,Q4,2001-Q4,Night,Motor Vehicle Theft,1
4,2001-12-31 20:00:00,PREDATORY,APARTMENT,0,0,1723,17,33,14,2001,60625,105830918.044,Albany Park,53542230.819,17,2,1723,14,Albany Park,53542230.8191,"(41.964286983,-87.713386478)",December,Monday,Q4,2001-Q4,Night,Criminal Sexual Assault,1


#### Convert to string
- Add padding if required

In [41]:
# change to string and must be three char length
cols = ['district', 'p_district', 'beat', 'p_beat', 'ward', 'community_area', 'p_sector', 'year', 'zip_code']

# iterate cols
for col in cols:

    if col in ['district', 'p_district']:
        # Fill NAs and convert to a standard string for the zfill operation
        # https://www.chicagopolice.org/statistics-data/crime-statistics/
        df_crime[col] = df_crime[col].astype("string").str.zfill(3)
    elif col in ['ward', 'community_area']:
        df_crime[col] = df_crime[col].astype("string").str.zfill(2)
    else:
        # Fill NAs and convert to a standard string
        df_crime[col] = df_crime[col].astype("string")
        
    # Force ArrowDtype
    df_crime[col] = df_crime[col].astype(arrow_string)

df_crime[cols].sample(5)

,district,p_district,beat,p_beat,ward,community_area,p_sector,year,zip_code
2098510,025,025,2514,2514,31,19,1,2010,60641
7424889,015,015,1511,1511,29,25,1,2022,60651
933192,009,009,915,915,03,34,1,2015,60609
7641887,007,007,725,725,16,67,2,2023,60636
1779680,009,009,914,924,12,61,2,2011,60609


### NaNs & Invalid Data

#### Community

In [42]:
print(df_crime.columns)

Index(['date', 'description', 'location_description', 'arrest', 'domestic',
       'beat', 'district', 'ward', 'community_area', 'year', 'zip_code',
       'zip_code_area', 'primary_neighborhood', 'neighborhood_area',
       'p_district', 'p_sector', 'p_beat', 'community_code', 'community_name',
       'ca_community_area', 'location', 'month', 'day_of_week', 'quarter',
       'year_quarter', 'time_of_day', 'fbi_code_desc', 'fbi_index_code'],
      dtype='object')


In [43]:
df_crime.community_area.isna().sum(), df_crime.community_code.isnull().sum(), df_crime.community_name.isnull().sum()

(np.int64(613692), np.int64(116237), np.int64(116237))

In [44]:
# Cross-column validation to handle missing community area data (Crime)
mask = (df_crime.community_area.isna() & df_crime.community_code.notna()
       & (df_crime.primary_neighborhood == df_crime.community_name)
       )
print(f"Mis-match count: {(mask.sum()):,}\n")
df_crime.loc[mask, ['community_area','primary_neighborhood', 'community_code', 'community_name', 'zip_code']].sample(10)

Mis-match count: 474,776



,community_area,primary_neighborhood,community_code,community_name,zip_code
245733,<NA>,Austin,25,Austin,60644
116083,<NA>,Fuller Park,37,Fuller Park,60609
6007577,<NA>,Woodlawn,42,Woodlawn,60637
7095066,<NA>,Ashburn,70,Ashburn,60652
51083,<NA>,New City,61,New City,60609
2990968,<NA>,Austin,25,Austin,60644
410884,<NA>,Austin,25,Austin,60639
214080,<NA>,New City,61,New City,60609
366659,<NA>,South Shore,43,South Shore,60649
399212,<NA>,Washington Heights,73,Washington Heights,60628


In [45]:
# Update community_area NaNs
df_crime.loc[mask, 'community_area'] = df_crime.loc[mask, 'community_code']

In [46]:
# Check Update
df_crime.loc[mask, ['community_area','primary_neighborhood', 'community_code', 'community_name', 'zip_code']].sample(5)

,community_area,primary_neighborhood,community_code,community_name,zip_code
292613,25,Austin,25,Austin,60644
5956719,3,Uptown,3,Uptown,60640
195996,49,Roseland,49,Roseland,60628
210119,14,Albany Park,14,Albany Park,60625
158181,25,Austin,25,Austin,60651


In [47]:
# Lets tackle the rest of the community area
# Determine if one is missing, the other is not
mask = df_crime.community_area.isna() ^ df_crime.community_code.isna()
print(f"Mis-match count: {(mask.sum()):,}\n")
df_crime.loc[mask, ['community_area','primary_neighborhood', 'community_code', 'community_name', 'zip_code']].sample(10)

Mis-match count: 235,659



,community_area,primary_neighborhood,community_code,community_name,zip_code
18553,49,<NA>,<NA>,<NA>,<NA>
832704,32,<NA>,<NA>,<NA>,<NA>
3865574,64,<NA>,<NA>,<NA>,<NA>
6011434,<NA>,Grand Crossing,69,Greater Grand Crossing,60637
856488,76,<NA>,<NA>,<NA>,<NA>
8000539,24,<NA>,<NA>,<NA>,<NA>
6890066,29,<NA>,<NA>,<NA>,<NA>
305333,<NA>,Englewood,67,West Englewood,60636
833282,28,<NA>,<NA>,<NA>,<NA>
202925,<NA>,Old Town,8,Near North Side,60610


In [48]:
#Cross-column validation to handle missing community area data (Crime)
mask = (df_crime.community_area.isna() & df_crime.community_code.notnull()
       & (df_crime.primary_neighborhood != df_crime.community_name)
       )
print(f"Mis-match count: {(mask.sum()):,}\n")
df_crime.loc[mask, ['community_area','primary_neighborhood', 'community_code', 'community_name', 'zip_code']].sort_values('zip_code').drop_duplicates().head(10)

Mis-match count: 129,169



,community_area,primary_neighborhood,community_code,community_name,zip_code
54117,<NA>,Millenium Park,32,Loop,60602
58402,<NA>,Millenium Park,32,Loop,60603
123519,<NA>,Grant Park,32,Loop,60603
44129,<NA>,Grant Park,32,Loop,60604
43787,<NA>,Grant Park,32,Loop,60605
43794,<NA>,Printers Row,32,Loop,60605
46251,<NA>,Museum Campus,33,Near South Side,60605
44922,<NA>,West Loop,28,Near West Side,60606
54327,<NA>,River North,8,Near North Side,60606
43529,<NA>,"Little Italy, UIC",28,Near West Side,60607


In [49]:
df_crime.loc[mask, ['community_area','primary_neighborhood', 'community_code', 'community_name', 'zip_code']] \
    .sort_values('zip_code').drop_duplicates().tail(10)

,community_area,primary_neighborhood,community_code,community_name,zip_code
43586,<NA>,River North,8,Near North Side,60654
50790,<NA>,Rush & Division,8,Near North Side,60654
44744,<NA>,O'Hare,76,Ohare,60656
47437,<NA>,Boystown,6,Lake View,60657
47851,<NA>,Wrigleyville,6,Lake View,60657
50455,<NA>,"Sauganash,Forest Glen",12,Forest Glen,60659
43588,<NA>,Greektown,28,Near West Side,60661
44002,<NA>,West Loop,28,Near West Side,60661
2282292,<NA>,O'Hare,76,Ohare,60666
43647,<NA>,Galewood,25,Austin,60707


In [50]:
# Update community_area with community_code
df_crime.loc[mask, 'community_area'] = df_crime.loc[mask, 'community_code']
# Update community_name Ohare to O'hare
df_crime['community_name'] = df_crime['community_name'].replace({'Ohare':"O'Hare"})

In [51]:
df_crime.loc[mask, ['community_area','primary_neighborhood', 'community_code', 'community_name', 'zip_code']].sort_values('zip_code').drop_duplicates().tail(10)

,community_area,primary_neighborhood,community_code,community_name,zip_code
43586,8,River North,8,Near North Side,60654
50790,8,Rush & Division,8,Near North Side,60654
44744,76,O'Hare,76,O'Hare,60656
47437,6,Boystown,6,Lake View,60657
47851,6,Wrigleyville,6,Lake View,60657
50455,12,"Sauganash,Forest Glen",12,Forest Glen,60659
43588,28,Greektown,28,Near West Side,60661
44002,28,West Loop,28,Near West Side,60661
2282292,76,O'Hare,76,O'Hare,60666
43647,25,Galewood,25,Austin,60707


In [52]:
# Columns List
cols = ['community_area','primary_neighborhood', 'community_code', 'community_name', 
        'district', 'beat', 'ward', 'zip_code']
# NaNs
mask = df_crime.community_area.isna()
df_crime.loc[mask, cols].sample(10)

,community_area,primary_neighborhood,community_code,community_name,district,beat,ward,zip_code
7083647,<NA>,<NA>,<NA>,<NA>,005,511,<NA>,<NA>
290037,<NA>,<NA>,<NA>,<NA>,016,1613,<NA>,<NA>
8208524,<NA>,<NA>,<NA>,<NA>,020,2023,<NA>,<NA>
222040,<NA>,<NA>,<NA>,<NA>,012,1212,<NA>,<NA>
144867,<NA>,<NA>,<NA>,<NA>,022,2213,<NA>,<NA>
41619,<NA>,<NA>,<NA>,<NA>,012,1323,<NA>,<NA>
9397,<NA>,<NA>,<NA>,<NA>,011,1111,27,<NA>
41439,<NA>,<NA>,<NA>,<NA>,003,312,<NA>,<NA>
41744,<NA>,<NA>,<NA>,<NA>,015,1512,<NA>,<NA>
40818,<NA>,<NA>,<NA>,<NA>,011,1113,<NA>,<NA>


In [53]:
mask = (df_crime.primary_neighborhood.isna() & df_crime.community_name.notna())
print(f"NaN count: {(mask.sum()):,}\n")

NaN count: 0



#### Police District

In [54]:
# columns to display
cols = ['ward', 'community_area', 'community_code', 'beat', 'p_beat', 'district', 'p_district',
        'p_sector', 'zip_code', 'primary_neighborhood', 'community_name']
# Invalid Police Districts
print(f"District 021 count:  {(df_crime.district.eq('021').sum()):,}")
print(f"District 031 count:  {(df_crime.district.eq('031').sum()):,}")

District 021 count:  4
District 031 count:  277


##### District 021

In [55]:
# For district 021
if (df_crime.district == '021').sum() <= 10:
    out = df_crime.loc[df_crime["district"].eq('021'), cols]
else:
    out = df_crime.loc[df_crime["district"].eq('021'), cols].sample(n=10, random_state=SEED)
# display
out

,ward,community_area,community_code,beat,p_beat,district,p_district,p_sector,zip_code,primary_neighborhood,community_name
5167447,03,35,35,2112,211,021,002,1,60616,Douglas,Douglas
5199214,03,35,35,2112,211,021,002,1,60616,Douglas,Douglas
5325554,03,35,35,2112,211,021,002,1,60616,Douglas,Douglas
5522033,03,35,35,2112,211,021,002,1,60616,Douglas,Douglas


In [56]:
# Update invalid district 021
mask = df_crime["district"].eq('021')
df_crime.loc[mask, "district"] = df_crime.loc[mask, "p_district"]

##### District 031

In [57]:
# For district 031
if (df_crime.district == '031').sum() <= 10:
    out = df_crime.loc[df_crime["district"].eq('031'), cols]
else:
    out = df_crime.loc[df_crime["district"].eq('031'), cols].sample(n=10, random_state=SEED)
# display
out

,ward,community_area,community_code,beat,p_beat,district,p_district,p_sector,zip_code,primary_neighborhood,community_name
7161373,41,76,<NA>,1654,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
6819956,41,76,<NA>,1653,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
3430409,06,69,<NA>,323,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
7428007,41,76,<NA>,1654,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
6251690,<NA>,<NA>,<NA>,533,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
6873037,41,76,<NA>,1654,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
6247970,<NA>,<NA>,<NA>,1651,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
3351498,06,69,<NA>,323,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
3937969,19,75,75,2212,2212,031,022,1,60655,Morgan Park,Morgan Park
4903971,19,75,75,2212,2212,031,022,1,60655,Morgan Park,Morgan Park


In [58]:
# Update invalid district 031
mask = df_crime["district"].eq('031')
df_crime.loc[mask, "district"] = df_crime.loc[mask, "p_district"]
# Update rest of invalid district 031 to NaNs
df_crime.loc[df_crime["district"] == '031', "district"] = pd.NA

In [59]:
# display district
utils.wrap_unique(df_crime, 'district')

[001, 002, 003, 004, 005, 006, 007, 008, 009, 010, 011, 012, 014, 015, 016, 017,
018, 019, 020, 022, 024, 025]
::::: Unique Count: 22 (+ 186 nulls)


##### Police District NaNs

In [60]:
# Cross columns check
mask = (df_crime.district.isna()) & df_crime.p_district.notnull()
print(f"Mis-match count: {(mask.sum()):,}\n")
df_crime.loc[mask, cols].sample(n=5, random_state=SEED)

Mis-match count: 47



,ward,community_area,community_code,beat,p_beat,district,p_district,p_sector,zip_code,primary_neighborhood,community_name
3824458,17,67,67,734,734,<NA>,007,3,60636,Englewood,West Englewood
3833083,20,40,40,234,232,<NA>,002,3,60637,Washington Park,Washington Park
3823732,29,25,25,2531,2531,<NA>,025,3,60651,Austin,Austin
2391225,47,05,5,1912,1911,<NA>,019,1,60618,North Center,North Center
3860783,39,14,14,1722,1722,<NA>,017,2,60630,Albany Park,Albany Park


In [61]:
# Update District
df_crime.loc[mask, 'district'] = df_crime.loc[mask, 'p_district'] 
# Validate
df_crime.loc[mask, cols].sample(n=5, random_state=SEED)

,ward,community_area,community_code,beat,p_beat,district,p_district,p_sector,zip_code,primary_neighborhood,community_name
3824458,17,67,67,734,734,007,007,3,60636,Englewood,West Englewood
3833083,20,40,40,234,232,002,002,3,60637,Washington Park,Washington Park
3823732,29,25,25,2531,2531,025,025,3,60651,Austin,Austin
2391225,47,05,5,1912,1911,019,019,1,60618,North Center,North Center
3860783,39,14,14,1722,1722,017,017,2,60630,Albany Park,Albany Park


In [62]:
# list of columns to display
cols = ['district', 'primary_neighborhood', 'community_code', 'community_name', 'p_sector', 'beat', 'zip_code']
mask = df_crime.district.isna()
print(f"NaN count: {(mask.sum()):,}\n")
df_crime.loc[mask, cols].sample(n=10, random_state=SEED)

NaN count: 139



,district,primary_neighborhood,community_code,community_name,p_sector,beat,zip_code
6780034,<NA>,<NA>,<NA>,<NA>,<NA>,1653,<NA>
7192131,<NA>,<NA>,<NA>,<NA>,<NA>,1654,<NA>
7399686,<NA>,<NA>,<NA>,<NA>,<NA>,1653,<NA>
3435476,<NA>,<NA>,<NA>,<NA>,<NA>,323,<NA>
1887188,<NA>,West Ridge,2,West Ridge,<NA>,2412,60645
7428007,<NA>,<NA>,<NA>,<NA>,<NA>,1654,<NA>
6743509,<NA>,<NA>,<NA>,<NA>,<NA>,1653,<NA>
3395739,<NA>,<NA>,<NA>,<NA>,<NA>,323,<NA>
2152276,<NA>,<NA>,<NA>,<NA>,<NA>,1654,<NA>
6861737,<NA>,<NA>,<NA>,<NA>,<NA>,1653,<NA>


In [63]:
# Finding Unique Key for Update
key_cols = ['primary_neighborhood',	'community_code', 'community_name',	'p_sector', 'beat', 'zip_code']
target_col = 'district'
# fill district using Composite key
df_crime = geo.fill_from_composite_key(df_crime, key_cols, target_col).copy()

--- Fill Summary: Column (district) 0 missing values filled ---


In [64]:
# Create new district_location column
df_crime['district_location'] = df_crime['district'].map(geo_dict.cpd_districts).astype(arrow_string)

# Validate unmapped districts
df_crime.loc[df_crime["district_location"].isna(), "district"].unique()

<ArrowExtensionArray>
[<NA>]
Length: 1, dtype: string[pyarrow]

#### Sector Mapping

In [65]:
# Invert the mapping
cpd_sector = {
    dist: sector
    for sector, dists in geo_dict.cpd_sector.items() # Outer loop: looping over sector → list_of_districts
    for dist in dists # Inner loop: looping over each district inside that list
}

# Create the new sector column
df_crime["map_sector"] = df_crime["district"].map(cpd_sector).astype(arrow_string)

# Validate unmapped districts
df_crime.loc[df_crime["map_sector"].isna(), "district"].unique()

<ArrowExtensionArray>
[<NA>]
Length: 1, dtype: string[pyarrow]

In [66]:
# columns to display
cols = ['p_beat', 'district', 'p_district', 'p_sector', 'map_sector',
        'zip_code', 'primary_neighborhood', 'community_name']
# Cross-reference check - looking for disagreement
mask = (df_crime.p_sector.isna() ^ df_crime.map_sector.isna())
print(f"Disagree count: {mask.sum()}")
df_crime.loc[mask, cols].drop_duplicates().sample(n=10, random_state=SEED)

Disagree count: 116671


,p_beat,district,p_district,p_sector,map_sector,zip_code,primary_neighborhood,community_name
127864,<NA>,001,<NA>,<NA>,3,60602,Loop,Loop
6839,<NA>,005,<NA>,<NA>,2,<NA>,<NA>,<NA>
5500,<NA>,012,<NA>,<NA>,3,<NA>,<NA>,<NA>
9399,<NA>,004,<NA>,<NA>,2,<NA>,<NA>,<NA>
2259,<NA>,014,<NA>,<NA>,5,<NA>,<NA>,<NA>
982237,<NA>,016,<NA>,<NA>,5,60631,<NA>,<NA>
7767176,<NA>,016,<NA>,<NA>,5,60646,"Sauganash,Forest Glen",Forest Glen
2267,<NA>,015,<NA>,<NA>,4,<NA>,<NA>,<NA>
9397,<NA>,011,<NA>,<NA>,4,<NA>,<NA>,<NA>
1072242,<NA>,024,<NA>,<NA>,3,60645,<NA>,<NA>


### DataFrame Maint

In [67]:
# Drop Columns
df_crime = df_crime.drop(columns=['p_beat', 'p_district', 'community_code', 'location', 'p_sector'])
# Rename Columns
df_crime = df_crime.rename(columns={'map_sector': 'sector', 'community_area': 'community_code',
                                    'ca_community_area' : 'community_area', 'primary_neighborhood': 'neighborhood'} )
# collapse fragmented blocks
df_crime = df_crime.copy()
# display
df_crime.head()

,date,description,location_description,arrest,domestic,beat,district,ward,community_code,year,zip_code,zip_code_area,neighborhood,neighborhood_area,community_name,community_area,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code,district_location,sector
0,2001-12-30 02:30:00,AUTOMOBILE,STREET,0,0,334,003,07,43,2001,60649,80526075.8505,South Shore,81812716.3904,South Shore,81812716.3958,December,Sunday,Q4,2001-Q4,Late Night,Motor Vehicle Theft,1,Grand Crossing,1
1,2001-12-24 12:30:00,FRAUD OR CONFIDENCE GAME,APARTMENT,0,0,2313,019,46,03,2001,60640,77305245.6358,Uptown,65095642.836,Uptown,65095642.7289,December,Monday,Q4,2001-Q4,Afternoon,Fraud,0,Town Hall,3
2,2001-12-31 23:00:00,FINANCIAL ID THEFT:$300 &UNDER,APARTMENT,0,0,1531,015,37,25,2001,60651,99039621.5713,Austin,170037750.826,Austin,199254203.427,December,Monday,Q4,2001-Q4,Night,Larceny – Theft,1,Austin,4
3,2001-12-23 22:30:00,AUTOMOBILE,STREET,0,0,1022,010,12,30,2001,60623,155285530.844,Little Village,127998297.819,South Lawndale,127998297.867,December,Sunday,Q4,2001-Q4,Night,Motor Vehicle Theft,1,Ogden,4
4,2001-12-31 20:00:00,PREDATORY,APARTMENT,0,0,1723,017,33,14,2001,60625,105830918.044,Albany Park,53542230.819,Albany Park,53542230.8191,December,Monday,Q4,2001-Q4,Night,Criminal Sexual Assault,1,Albany Park,5


In [68]:
df_crime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469872 entries, 0 to 8469871
Data columns (total 25 columns):
 #   Column                Dtype                          
---  ------                -----                          
 0   date                  timestamp[s][pyarrow]          
 1   description           string[pyarrow]                
 2   location_description  string[pyarrow]                
 3   arrest                extension<arrow.bool8>[pyarrow]
 4   domestic              extension<arrow.bool8>[pyarrow]
 5   beat                  string[pyarrow]                
 6   district              string[pyarrow]                
 7   ward                  string[pyarrow]                
 8   community_code        string[pyarrow]                
 9   year                  string[pyarrow]                
 10  zip_code              string[pyarrow]                
 11  zip_code_area         double[pyarrow]                
 12  neighborhood          string[pyarrow]                
 1

In [69]:
# NaNs
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,872) ---
                       Count Percentage
ward                  614815    7.2588%
neighborhood          116237    1.3724%
neighborhood_area     116237    1.3724%
community_name        116237    1.3724%
community_area        116237    1.3724%
zip_code              116181    1.3717%
zip_code_area         116181    1.3717%
location_description   15338    0.1811%
community_code          9747    0.1151%
district                 139    0.0016%
district_location        139    0.0016%
sector                   139    0.0016%


#### Neighborhood & Community

In [70]:
# list of columns to display
cols = ['neighborhood', 'neighborhood_area', 'community_code', 'community_name', 'community_area', 'zip_code', 'zip_code_area']
# Only zip codes not null
mask = df_crime.zip_code.notna() & (df_crime.neighborhood.isna() | df_crime.community_name.isna())
print(f"NaN count: {(mask.sum()):,}\n")
df_crime.loc[mask, cols].drop_duplicates().sort_values('zip_code')

NaN count: 71



,neighborhood,neighborhood_area,community_code,community_name,community_area,zip_code,zip_code_area
982237,<NA>,<NA>,10,<NA>,<NA>,60631,107117592.044
1072242,<NA>,<NA>,02,<NA>,<NA>,60645,62181473.4847
3351185,<NA>,<NA>,72,<NA>,<NA>,60655,115380139.518
147164,<NA>,<NA>,<NA>,<NA>,<NA>,60656,89515884.2795
1117332,<NA>,<NA>,10,<NA>,<NA>,60656,89515884.2795
1164985,<NA>,<NA>,76,<NA>,<NA>,60656,89515884.2795


**Note:**
- Inconsistency in zip_code with related columns
- Only Update NaNs: neighborhood & community_name

In [71]:
# Determine Mapping
df_crime[['neighborhood', 'community_name', 'zip_code', 'zip_code_area']] \
    [(df_crime.zip_code.isin(['60631', '60645', '60655', '60656'  ]))] \
    .drop_duplicates().dropna().sort_values('zip_code')

,neighborhood,community_name,zip_code,zip_code_area
478,O'Hare,O'Hare,60631,107117592.044
489,Norwood Park,Norwood Park,60631,107117592.044
1077,Edison Park,Edison Park,60631,107117592.044
94,West Ridge,West Ridge,60645,62181473.4847
630,Rogers Park,Rogers Park,60645,62181473.4847
47,Mount Greenwood,Mount Greenwood,60655,115380139.518
643,Morgan Park,Morgan Park,60655,115380139.518
3826,Beverly,Beverly,60655,115380139.518
206,O'Hare,O'Hare,60656,89515884.2795
284,Norwood Park,Norwood Park,60656,89515884.2795


##### Neighborhood & Community_name

In [72]:
# mapping table
mapping = {'60631': 'Norwood Park', '60645': 'West Ridge', '60655': 'Mount Greenwood', '60656': "O'Hare"}

# Ensure Arrow string dtype for speed
zip_col = df_crime["zip_code"].astype(arrow_string)

# Build the mapped values
mapped_zip = zip_col.map(mapping)

# Mask: rows where neighborhood OR community_name is missing
nc_mask = df_crime["neighborhood"].isna() | df_crime["community_name"].isna()

# Apply mapping only to masked rows, keep original otherwise
df_crime.loc[nc_mask, "neighborhood"] = (
    mapped_zip.fillna(df_crime.loc[nc_mask, "neighborhood"])
)

df_crime.loc[nc_mask, "community_name"] = (
    mapped_zip.fillna(df_crime.loc[nc_mask, "community_name"])
)

In [73]:
# NaNs
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,872) ---
                       Count Percentage
ward                  614815    7.2588%
neighborhood_area     116237    1.3724%
community_area        116237    1.3724%
zip_code              116181    1.3717%
zip_code_area         116181    1.3717%
neighborhood          116166    1.3715%
community_name        116166    1.3715%
location_description   15338    0.1811%
community_code          9747    0.1151%
district                 139    0.0016%
district_location        139    0.0016%
sector                   139    0.0016%


##### Community Code

In [74]:
# list of columns to display
cols = ['neighborhood', 'neighborhood_area', 'community_code', 'community_name', 'community_area', 'zip_code', 'zip_code_area']
# Only community codes null
mask = df_crime.community_code.notna() & (df_crime.neighborhood.isna() | df_crime.community_name.isnull())
print(f"NaN count: {(mask.sum()):,}\n")
df_crime.loc[mask, cols].drop_duplicates().sort_values('zip_code').sample(n=10, random_state=SEED)

NaN count: 106,422



,neighborhood,neighborhood_area,community_code,community_name,community_area,zip_code,zip_code_area
9380,<NA>,<NA>,19,<NA>,<NA>,<NA>,<NA>
2787,<NA>,<NA>,36,<NA>,<NA>,<NA>,<NA>
9471,<NA>,<NA>,61,<NA>,<NA>,<NA>,<NA>
9457,<NA>,<NA>,26,<NA>,<NA>,<NA>,<NA>
9422,<NA>,<NA>,57,<NA>,<NA>,<NA>,<NA>
3710,<NA>,<NA>,04,<NA>,<NA>,<NA>,<NA>
9473,<NA>,<NA>,38,<NA>,<NA>,<NA>,<NA>
6839,<NA>,<NA>,54,<NA>,<NA>,<NA>,<NA>
9445,<NA>,<NA>,60,<NA>,<NA>,<NA>,<NA>
9406,<NA>,<NA>,45,<NA>,<NA>,<NA>,<NA>


In [75]:
# Fill NaNs for neighborhood & community_name using community_code
df_crime = geo.fill_geo_from_lookup(df_crime, 'community_code', ['neighborhood', 'community_name']).copy()

--- Fill Summary (Key: community_code) ---
Column 'neighborhood': 106,422 rows filled.
Column 'community_name': 106,422 rows filled.


In [76]:
# Fill NaNs for zip_code using community_code
df_crime = geo.fill_geo_from_lookup(df_crime, 'community_code', ['zip_code']).copy()

--- Fill Summary (Key: community_code) ---
Column 'zip_code': 106,437 rows filled.


In [77]:
# Replaces : , -, and multi-spaces with a single space
def clean_locations(series):
    out = (
        series.str.replace(r'[\s:,,-]+', ' ', regex=True) # Combined delimiters to space
              .str.replace(r'\s*/\s*', '/', regex=True)  # Fix slashes
              .str.strip()
    )
    
    return out
# Apply regex
df_crime['location_description'] = clean_locations(df_crime['location_description'])

# Dictionary mapping (Vectorized replace)
mapping = {
    'NURSING HOME/RETIREMENT HOME': 'NURSING/RETIREMENT HOME', 
    'OTHER RAILROAD PROP/TRAIN DEPOT': 'OTHER RAILROAD PROPERTY/TRAIN DEPOT',
    'PARKING LOT/GARAGE(NON.RESID.)': 'PARKING LOT/GARAGE (NON RESIDENTIAL)',
    'POLICE FACILITY/VEH PARKING LOT': 'POLICE FACILITY/VEHICLE PARKING LOT',
    'POOLROOM': 'POOL ROOM', 
    'RESIDENCE YARD (FRONT/BACK)': 'RESIDENTIAL YARD (FRONT/BACK)',
    'TAXICAB': 'TAXI CAB',
    'VEHICLE OTHER RIDE SERVICE': 'VEHICLE OTHER RIDE SHARE SERVICE (LYFT UBER ETC.)',
    'VEHICLE OTHER RIDE SHARE SERVICE (E.G. UBER LYFT)': 'VEHICLE OTHER RIDE SHARE SERVICE (LYFT UBER ETC.)'
}

# Apply mapping first
df_crime['location_description'] = df_crime['location_description'].replace(mapping)

In [78]:
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,872) ---
                       Count Percentage
ward                  614815    7.2588%
neighborhood_area     116237    1.3724%
community_area        116237    1.3724%
zip_code_area         116181    1.3717%
location_description   15338    0.1811%
community_code          9747    0.1151%
zip_code                9744    0.1150%
neighborhood            9744    0.1150%
community_name          9744    0.1150%
district                 139    0.0016%
district_location        139    0.0016%
sector                   139    0.0016%


#### Ward

In [79]:
# list of columns to display
cols = ['ward', 'neighborhood', 'community_code', 'community_name', 'sector', 'district', 'beat', 'zip_code']
# Only Ward is NaN
mask = df_crime.ward.isna()
print(f"NaN count: {(mask.sum()):,}\n")
df_crime.loc[mask, cols].sample(n=10, random_state=SEED)

NaN count: 614,815



,ward,neighborhood,community_code,community_name,sector,district,beat,zip_code
6004400,<NA>,Woodlawn,42,Woodlawn,1,003,321,60637
420401,<NA>,Garfield Park,27,East Garfield Park,4,011,1135,60612
5994657,<NA>,Garfield Park,26,West Garfield Park,4,011,1114,60624
78620,<NA>,North Lawndale,29,North Lawndale,4,010,1012,60623
169089,<NA>,United Center,28,Near West Side,3,012,1211,60612
442483,<NA>,Ashburn,70,Ashburn,2,006,614,60620
262710,<NA>,Woodlawn,42,Woodlawn,1,003,321,60637
97641,<NA>,Garfield Park,27,East Garfield Park,3,012,1331,60612
6007261,<NA>,West Ridge,2,West Ridge,3,024,2413,60659
355372,<NA>,Lincoln Park,7,Lincoln Park,3,019,1931,60614


In [80]:
# Finding Unique Key for Update
key_cols = ['neighborhood',	'community_code', 'community_name',	'sector', 'district', 'beat', 'zip_code']
target_col = 'ward'
# fill ward using Composite key
df_crime = geo.fill_from_composite_key(df_crime, key_cols, target_col)

--- Fill Summary: Column (ward) 530,713 missing values filled ---


In [81]:
# check
df_crime.loc[mask, cols].sample(n=10, random_state=SEED)

,ward,neighborhood,community_code,community_name,sector,district,beat,zip_code
6004400,20,Woodlawn,42,Woodlawn,1,003,321,60637
420401,02,Garfield Park,27,East Garfield Park,4,011,1135,60612
5994657,28,Garfield Park,26,West Garfield Park,4,011,1114,60624
78620,24,North Lawndale,29,North Lawndale,4,010,1012,60623
169089,02,United Center,28,Near West Side,3,012,1211,60612
442483,18,Ashburn,70,Ashburn,2,006,614,60620
262710,20,Woodlawn,42,Woodlawn,1,003,321,60637
97641,27,Garfield Park,27,East Garfield Park,3,012,1331,60612
6007261,<NA>,West Ridge,2,West Ridge,3,024,2413,60659
355372,<NA>,Lincoln Park,7,Lincoln Park,3,019,1931,60614


**Note:**
- Check East Garfield Park - Data inconsistencies
- Probamatic due to redistricting that happens every 10 years after the U.S. Census, or possible data

In [82]:
# examine community_name equal East Garfield Park
df_crime.loc[df_crime["community_name"].eq("East Garfield Park")] \
    .groupby(["ward", "community_code", "community_name", "zip_code"]) \
    .size()

ward  community_code  community_name      zip_code
02    27              East Garfield Park  60612       34779
                                          60624         311
      28              East Garfield Park  60612         102
06    44              East Garfield Park  60624           1
24    26              East Garfield Park  60624         444
      27              East Garfield Park  60612        5724
                                          60624       21268
      29              East Garfield Park  60612          29
                                          60624         986
27    23              East Garfield Park  60612          54
                                          60624          13
      26              East Garfield Park  60624         229
      27              East Garfield Park  60612       13493
                                          60624         841
      28              East Garfield Park  60612           7
28    23              East Garfield Park  60612  

In [83]:
# Only community_code is null & community_name not null
mask = df_crime.community_code.isna() & df_crime.community_name.notna()
print(f"NaN count: {(mask.sum()):,}\n")
df_crime.loc[mask, cols].head()

NaN count: 3



,ward,neighborhood,community_code,community_name,sector,district,beat,zip_code
147164,<NA>,O'Hare,<NA>,O'Hare,5,016,1614,60656
294937,<NA>,O'Hare,<NA>,O'Hare,5,016,1614,60656
5953133,<NA>,O'Hare,<NA>,O'Hare,5,016,1613,60656


In [84]:
# Fill NaNs for neighborhood & community_name using community_code
df_crime = geo.fill_geo_from_lookup(df_crime, 'community_name', ['community_code']).copy()

--- Fill Summary (Key: community_name) ---
Column 'community_code': 3 rows filled.


In [85]:
# Finding Unique Key for Update
key_cols = ['neighborhood',	'community_code', 'community_name',	'sector', 'district', 'beat', 'zip_code']
target_col = 'ward'
# fill ward using Composite key
df_crime = geo.fill_from_composite_key(df_crime, key_cols, target_col)

--- Fill Summary: Column (ward) 2 missing values filled ---


In [86]:
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,872) ---
                       Count Percentage
neighborhood_area     116237    1.3724%
community_area        116237    1.3724%
zip_code_area         116181    1.3717%
ward                   84100    0.9929%
location_description   15338    0.1811%
community_code          9744    0.1150%
zip_code                9744    0.1150%
neighborhood            9744    0.1150%
community_name          9744    0.1150%
district                 139    0.0016%
district_location        139    0.0016%
sector                   139    0.0016%


## Total Time

In [87]:
# total elapsed time
elapsed = time.time() - start
print(f"{elapsed:.2f}s to process {df_crime.shape[0]:,} rows")

78.45s to process 8,469,872 rows


## Save using Native PyArrow IPC

In [88]:
# Convert Pandas DataFrame to PyArrow Table
table = pa.Table.from_pandas(df_crime)

# Save as Arrow IPC stream (faster than pickle)
with open('../Data/crime_data.arrow', 'wb') as f:
    with ipc.new_file(f, table.schema) as writer:
        writer.write_table(table)